<a href="https://colab.research.google.com/github/Peeyusj/rag_sratch/blob/main/first_langchain_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install chromadb sentence-transformers groq langchain langchain-community langchain-groq langchain-chroma langchain-core --quiet

In [2]:
import urllib.request

url = "https://www.gutenberg.org/cache/epub/19630/pg19630.txt"

with urllib.request.urlopen(url) as response:
    raw_text = response.read().decode('utf-8')

clean_text = raw_text[2275:284556]
print(len(clean_text))

282281


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ".", " "],
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_text(clean_text)
print(f"Total chunks: {len(chunks)}")

Total chunks: 610


In [4]:
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from google.colab import drive

drive.mount('/content/drive')

# Initialize embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")




# Load existing vectorstore instead of creating new one
vectorstore = Chroma(
    embedding_function=embeddings,
    persist_directory="/content/drive/MyDrive/chroma_langchain"
)

if vectorstore._collection.count() == 0:
    # This one line does what took you 10 lines manually:
    # — embeds all chunks
    # — stores in ChromaDB
    # — persists to Drive
    vectorstore = Chroma.from_texts(
        texts=chunks,
        embedding=embeddings,
        persist_directory="/content/drive/MyDrive/chroma_langchain"
    )
    print(f"Stored {vectorstore._collection.count()} chunks")
else:
    print(f"Collection already exists with {vectorstore._collection.count()} chunks — skipping")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_34396/2271839012.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Stored 610 chunks


In [5]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from google.colab import userdata

# Initialize Groq LLM
llm = ChatGroq(
    api_key=userdata.get('GROQ_API_KEY'),
    model_name="llama-3.1-8b-instant"
)

# Convert vectorstore into retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Prompt template — {context} and {question} get filled automatically
prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context below.

Context: {context}
Question: {question}
""")

# LCEL chain — this is the modern LangChain way
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

answer = chain.invoke("Who was Arjuna's greatest enemy?")
print(answer)

The answer cannot be determined based on the given context. The provided pages seem to be describing Arjuna's battles and victories, but they do not specifically mention his greatest enemy.
